# 02 – EDA: Google search data analyst postings (Luke Barousse)

Goal: check if this second dataset can add data job salaries to the LinkedIn data, and what biases it has.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

gs = pd.read_csv("../data/raw/gsearch/gsearch_jobs.csv")
gs.shape

(61953, 27)

In [2]:
gs = gs.drop(columns=["Unnamed: 0", "index"])

print("rows:", len(gs))
print("unique job_id:", gs["job_id"].nunique())

dup_cols = ["title", "company_name", "location"]
print("unique title+company+location:", len(gs.drop_duplicates(dup_cols)))

gs_sal = gs[gs["salary_standardized"].notna()]
print("salary rows:", len(gs_sal))
print("unique salary postings:", len(gs_sal.drop_duplicates(dup_cols)))

rows: 61953
unique job_id: 58775
unique title+company+location: 41719
salary rows: 10088
unique salary postings: 7226


In [3]:
print(gs["salary_rate"].value_counts(dropna=False))
print(gs["via"].value_counts().head(10))
gs_sal["salary_standardized"].describe().round(0)

salary_rate
NaN        51865
an hour     5900
a year      4069
a month      119
Name: count, dtype: int64
via
via LinkedIn        20475
via Upwork           7261
via BeBee            5378
via Trabajo.org      3017
BeBee                2937
via ZipRecruiter     2878
LinkedIn             1924
via Indeed           1849
Mysmartpros          1417
via Snagajob          970
Name: count, dtype: int64


count     10088.0
mean      92289.0
std       43278.0
min       15080.0
25%       62400.0
50%       88400.0
75%      117500.0
max      624000.0
Name: salary_standardized, dtype: float64

In [4]:
gs_u = gs_sal.drop_duplicates(dup_cols).copy()
gs_u["source"] = gs_u["via"].str.replace("via ", "", regex=False)

print(gs_u["schedule_type"].value_counts(dropna=False))

(gs_u.groupby("source")["salary_standardized"]
     .agg(["count", "median"])
     .sort_values("count", ascending=False)
     .head(10))

schedule_type
Full-time                               3310
Contractor                              2399
Contractor and Temp work                1008
Full-time and Part-time                  222
Part-time                                 77
Full-time, Part-time, and Internship      48
Full-time and Contractor                  31
NaN                                       25
Full-time, Contractor, and Temp work      25
Part-time, Contractor, and Temp work      18
Temp work                                 16
Internship                                13
Full-time and Internship                   9
Full-time and Temp work                    9
Full-time, Part-time, and Contractor       8
Part-time and Contractor                   5
Part-time and Full-time                    2
Full-time, Part-time, and Per diem         1
Name: count, dtype: int64


,count,median
source,,
Upwork,2821,69680.0
LinkedIn,1624,107310.0
Indeed,647,85000.0
ZipRecruiter,278,96500.0
Ai-Jobs.net,260,108313.5
Snagajob,256,48391.2
BeBee,246,85000.0
Built In,194,107500.0
Ladders,122,104550.0


In [5]:
gs_u.groupby(["source", "salary_rate"]).size().unstack(fill_value=0).loc[
    ["LinkedIn", "Upwork", "BeBee", "Indeed", "ZipRecruiter"]]

salary_rate,a month,a year,an hour
source,,,
LinkedIn,14,902,708
Upwork,0,0,2821
BeBee,0,246,0
Indeed,6,434,207
ZipRecruiter,0,251,27


In [6]:
full_time = gs_u["schedule_type"].str.contains("Full-time", na=False)
not_gig = ~gs_u["source"].isin(["Upwork", "Snagajob"])

gs_clean = gs_u[full_time & not_gig].copy()
print("clean salary postings:", len(gs_clean))
print(gs_clean["salary_rate"].value_counts())
gs_clean["salary_standardized"].describe().round(0)

clean salary postings: 3358
salary_rate
a year     2840
an hour     492
a month      26
Name: count, dtype: int64


count      3358.0
mean     102828.0
std       37935.0
min       20800.0
25%       77518.0
50%       97400.0
75%      120640.0
max      550000.0
Name: salary_standardized, dtype: float64

### Cleaning rules (gsearch)

- Dedup on title + company + location: 10,088 → 7,226 salary postings.
- Drop Upwork (100% hourly freelance, 39% of salary rows) and Snagajob (hourly gigs).
- Keep only schedule_type containing "Full-time".

In [7]:
import sys
sys.path.append("..")
from src.roles import role_family

gs_clean["role"] = gs_clean["title"].apply(role_family)
gs_clean.groupby("role")["salary_standardized"].agg(["count", "median"]).round(0)

,count,median
role,,
BI Analyst,196,99400.0
Data Analyst,2538,95000.0
Data Engineer,186,125000.0
Data Manager,92,104250.0
Data Scientist,247,135200.0
ML/AI Engineer,27,132000.0
Other,72,98985.0


In [8]:
STATE_RE = r",\s*([A-Z]{2})\b"
gs_clean["state"] = gs_clean["location"].str.extract(STATE_RE)[0]
print("no state:", gs_clean["state"].isna().sum())
gs_clean["state"].value_counts(normalize=True).head(10).round(3)

no state: 2381


state
MO    0.240
OK    0.214
CO    0.213
KS    0.202
AR    0.082
NM    0.019
WY    0.011
CA    0.009
NE    0.004
TX    0.003
Name: proportion, dtype: float64

### Selection bias between the two datasets

- LinkedIn data salaries: 44% from CA, NY, WA (pay transparency laws, high-cost markets).
- Google search salaries: 71% have no state; of the rest, 87% from MO, OK, CO, KS, AR.
- This explains why gsearch is 13–20% lower within the same role.
- Decision: MVP salary model on LinkedIn (35,910 rows, state as feature). Add gsearch later only if it lowers test error on LinkedIn data jobs.